# NLP Masterclass – Análisis de Planes de Gobierno
## TF-IDF, Word2Vec, Topic Modeling, Similaridad Semántica y Clustering

**Curso:** Machine Learning  
**Autor del curso:** Josef Rodriguez

---

## Introducción

En este notebook construiremos una **Masterclass completa de Procesamiento de Lenguaje Natural (NLP)** usando un dataset real de **planes de gobierno** de candidatos presidenciales.

La idea no es solo aplicar código, sino **entender cada concepto**, su lógica, sus ventajas, sus limitaciones y cómo se integra dentro de un pipeline profesional de análisis de texto.

Trabajaremos con un dataset en español almacenado en el repositorio del curso:

`https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv`

---

## Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

- entender qué es NLP y por qué es importante
- limpiar y preparar texto en español
- representar documentos con **TF-IDF**
- aprender embeddings con **Word2Vec**
- construir embeddings de documentos
- comparar candidatos mediante **similaridad del coseno**
- visualizar estructuras semánticas con **PCA** y **t-SNE**
- descubrir temas latentes con **LDA (Topic Modeling)**
- generar **WordClouds**
- aplicar **clustering con K-Means**
- interpretar resultados de manera crítica

---

## Pipeline que construiremos

Texto → Limpieza → Tokenización → EDA → TF-IDF → Word2Vec → Embeddings de documentos → Similaridad → PCA / t-SNE → LDA → Clustering

---

## ¿Por qué este dataset es potente para una clase?

Porque permite trabajar con:

- **texto real**
- **español**
- **política y discurso público**
- **documentos largos**
- **comparación semántica entre actores**

Esto hace que la clase sea más cercana a proyectos reales de NLP que un dataset pequeño de juguete.


# 1. Librerías

Usaremos varias librerías especializadas:

- **pandas**: manipulación tabular
- **numpy**: operaciones numéricas
- **matplotlib** y **seaborn**: visualización
- **scikit-learn**: TF-IDF, PCA, t-SNE, K-Means, similitud del coseno
- **gensim**: Word2Vec y LDA
- **wordcloud**: nubes de palabras
- **collections.Counter**: conteo de tokens

> Si alguna librería no está instalada, puedes usar `pip install nombre_libreria`.


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA, LatentDirichletAllocation
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import Word2Vec
from wordcloud import WordCloud

sns.set_theme()
pd.set_option("display.max_colwidth", 200)


# 2. Carga del dataset desde GitHub

El dataset contiene dos columnas principales:

- **presidente**: nombre del candidato
- **texto_plan**: texto completo del plan de gobierno

Cada fila representa un documento.


In [ ]:
url = "https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv"

df = pd.read_csv(url)

df.head()


# 3. Exploración inicial del dataset

Antes de modelar, siempre conviene entender la estructura del dato.

Preguntas básicas:

- ¿cuántos documentos tenemos?
- ¿hay valores nulos?
- ¿qué tan largos son los documentos?
- ¿todos los documentos parecen válidos?


In [ ]:
df.info()


In [ ]:
print("Número de candidatos:", len(df))
print("Nulos por columna:")
print(df.isna().sum())


In [ ]:
df["longitud_caracteres"] = df["texto_plan"].fillna("").str.len()

plt.figure(figsize=(9, 5))
sns.histplot(df["longitud_caracteres"], bins=20)
plt.title("Distribución de longitud de planes de gobierno (caracteres)")
plt.xlabel("Número de caracteres")
plt.ylabel("Frecuencia")
plt.show()


## Interpretación

La longitud de los documentos importa porque en NLP los planes muy cortos y muy largos pueden afectar:

- la densidad de términos
- la calidad del embedding promedio
- la comparación entre candidatos
- la inferencia de temas

En problemas reales, siempre conviene revisar este aspecto.


# 4. Limpieza de texto

## ¿Qué es la limpieza de texto?

La limpieza de texto es el proceso de transformar documentos crudos en una versión más consistente para análisis computacional.

En este notebook haremos una limpieza **conservadora**, porque queremos mantener la mayor parte del contenido semántico.

### Pasos

- convertir a minúsculas
- eliminar saltos de línea
- eliminar caracteres especiales y números
- normalizar espacios

> En una versión más avanzada podríamos agregar lematización, stemming o stopwords personalizadas.


In [ ]:
def limpiar_texto(texto: str) -> str:
    texto = str(texto).lower()
    texto = re.sub(r"\n", " ", texto)
    texto = re.sub(r"[^a-záéíóúñü ]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["texto_clean"] = df["texto_plan"].apply(limpiar_texto)

df[["presidente", "texto_clean"]].head(2)


# 5. Tokenización

## ¿Qué es un token?

Un **token** es una unidad mínima de análisis textual. En muchos problemas de NLP clásico, un token es una palabra.

Ejemplo:

`"la economía crecerá este año"` → `["la", "economía", "crecerá", "este", "año"]`

La tokenización es fundamental porque muchos algoritmos operan sobre listas de palabras y no sobre cadenas completas.


In [ ]:
df["tokens"] = df["texto_clean"].str.split()

df["num_palabras"] = df["tokens"].apply(len)

df[["presidente", "num_palabras"]].head()


In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(
    data=df.sort_values("num_palabras", ascending=False),
    x=df.sort_values("num_palabras", ascending=False)["presidente"].str.split().str[0],
    y="num_palabras"
)
plt.xticks(rotation=90)
plt.title("Cantidad de palabras por candidato")
plt.xlabel("Candidato")
plt.ylabel("Número de palabras")
plt.show()


# 6. Exploración léxica básica

Antes de pasar a modelos más complejos, conviene mirar las palabras más frecuentes.

Esto ayuda a detectar:

- ruido
- palabras dominantes
- vocabulario político recurrente
- necesidad de limpiar más el corpus


In [ ]:
all_tokens = [token for doc in df["tokens"] for token in doc]
frecuencias = Counter(all_tokens)

top_20 = pd.DataFrame(frecuencias.most_common(20), columns=["palabra", "frecuencia"])
top_20


In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=top_20, x="frecuencia", y="palabra")
plt.title("20 palabras más frecuentes del corpus")
plt.xlabel("Frecuencia")
plt.ylabel("Palabra")
plt.show()


# 7. WordCloud general del corpus

## ¿Qué es una WordCloud?

Una **nube de palabras** es una visualización donde el tamaño de cada palabra depende de su frecuencia o peso.

No reemplaza análisis cuantitativo, pero es útil como exploración inicial o recurso pedagógico.


In [ ]:
texto_total = " ".join(df["texto_clean"])

wc = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    collocations=False
).generate(texto_total)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud general de planes de gobierno")
plt.show()


# 8. TF-IDF

## Definición

**TF-IDF (Term Frequency – Inverse Document Frequency)** es una técnica clásica de representación de texto.

Busca medir qué tan importante es una palabra dentro de un documento considerando también cuán común es en todo el corpus.

### Intuición

- Si una palabra aparece mucho en un documento, probablemente sea importante para ese documento.
- Pero si aparece en todos los documentos, entonces no ayuda mucho a diferenciarlos.

### Fórmulas

\[
TF(t,d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}
\]

\[
IDF(t) = \log\left(\frac{N}{df_t}\right)
\]

\[
TFIDF(t,d) = TF(t,d) \times IDF(t)
\]

donde:

- \(t\): término
- \(d\): documento
- \(N\): número total de documentos
- \(df_t\): cantidad de documentos que contienen el término


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words="english"  # cambiar si prefieres manejar stopwords en español manualmente
)

X_tfidf = vectorizer.fit_transform(df["texto_clean"])

print("Shape TF-IDF:", X_tfidf.shape)


## Nota importante

`scikit-learn` no incluye por defecto una lista robusta de stopwords en español para todos los casos.

En una clase avanzada puedes:

- usar `nltk.corpus.stopwords`
- construir una lista propia
- combinar stopwords genéricas con stopwords de dominio político

En esta notebook mantenemos una versión simple para favorecer reproducibilidad.


In [ ]:
feature_names = vectorizer.get_feature_names_out()
scores = np.asarray(X_tfidf.mean(axis=0)).flatten()

top_idx = scores.argsort()[-25:]
top_terms = pd.DataFrame({
    "palabra": [feature_names[i] for i in top_idx],
    "score_tfidf": scores[top_idx]
}).sort_values("score_tfidf", ascending=False)

top_terms.head(15)


In [ ]:
plt.figure(figsize=(9, 7))
sns.barplot(data=top_terms.head(20), x="score_tfidf", y="palabra")
plt.title("Términos con mayor peso promedio TF-IDF")
plt.xlabel("Peso promedio")
plt.ylabel("Palabra")
plt.show()


# 9. TF-IDF por candidato

También podemos identificar las palabras más representativas de un candidato específico.

Esto es útil para responder preguntas como:

- ¿qué temas enfatiza más un candidato?
- ¿qué vocabulario lo diferencia?


In [ ]:
def top_tfidf_terms_for_doc(doc_index, top_n=15):
    row = X_tfidf[doc_index].toarray().flatten()
    idx = row.argsort()[-top_n:][::-1]
    return pd.DataFrame({
        "palabra": [feature_names[i] for i in idx],
        "score_tfidf": row[idx]
    })

doc_index = 0
print("Candidato:", df.loc[doc_index, "presidente"])
top_tfidf_terms_for_doc(doc_index, top_n=15)


# 10. Word2Vec

## Definición

**Word2Vec** es un modelo de aprendizaje no supervisado que aprende **embeddings de palabras**, es decir, vectores densos que capturan relaciones semánticas.

La idea central es:

> Palabras que aparecen en contextos similares tendrán vectores similares.

### ¿Por qué es importante?

A diferencia de TF-IDF, Word2Vec no solo mide importancia de términos; también intenta aprender **significado distribuido**.

### Arquitecturas principales

- **CBOW**: predice la palabra central a partir del contexto
- **Skip-Gram**: predice el contexto a partir de la palabra central

En `gensim`:

- `sg=0` → CBOW
- `sg=1` → Skip-Gram


In [ ]:
sentences = df["tokens"].tolist()

w2v = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=10,
    min_count=5,
    sg=1,
    workers=4,
    seed=42
)

print("Tamaño del vocabulario Word2Vec:", len(w2v.wv))


## Parámetros clave

- **vector_size**: dimensión del embedding
- **window**: tamaño del contexto
- **min_count**: frecuencia mínima para incluir una palabra
- **sg=1**: usa Skip-Gram, útil para capturar mejor palabras menos frecuentes


In [ ]:
palabras_consulta = ["economia", "salud", "educacion", "seguridad"]

for palabra in palabras_consulta:
    if palabra in w2v.wv:
        print(f"\nPalabras similares a '{palabra}':")
        print(w2v.wv.most_similar(palabra, topn=8))
    else:
        print(f"'{palabra}' no está en el vocabulario.")


# 11. Visualización de embeddings de palabras con PCA

Los embeddings tienen 100 dimensiones. Para visualizarlos, reducimos a 2 dimensiones.

## ¿Qué hace PCA?

**PCA (Principal Component Analysis)** proyecta los datos en nuevas dimensiones que explican la mayor parte de la varianza.

Es útil para tener una visión global, aunque no siempre separa bien grupos semánticos finos.


In [ ]:
selected_words = [w for w in ["economia", "salud", "educacion", "seguridad", "trabajo", "pobreza",
                              "inversion", "corrupcion", "estado", "empresa", "mujer", "niñez"] if w in w2v.wv]

word_vectors = np.array([w2v.wv[w] for w in selected_words])

pca_words = PCA(n_components=2, random_state=42)
coords_words_pca = pca_words.fit_transform(word_vectors)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=coords_words_pca[:, 0], y=coords_words_pca[:, 1])

for i, w in enumerate(selected_words):
    plt.text(coords_words_pca[i, 0], coords_words_pca[i, 1], w)

plt.title("PCA de embeddings de palabras seleccionadas")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


# 12. Embeddings de documentos

## Problema

Word2Vec produce vectores para **palabras**, no para documentos completos.

## Solución simple

Una forma clásica de representar un documento es **promediar los embeddings de sus palabras**.

No es la técnica más sofisticada, pero es útil, interpretable y muy buena para enseñar.


In [ ]:
def doc_embedding(tokens, model, dim=100):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

doc_vectors = np.vstack(df["tokens"].apply(lambda x: doc_embedding(x, w2v, dim=100)))

doc_vectors.shape


# 13. Similaridad entre candidatos

## Similaridad del coseno

La similitud del coseno mide el ángulo entre dos vectores.

\[
\text{cosine}(A,B)=\frac{A\cdot B}{||A||\,||B||}
\]

Interpretación:

- cercano a **1** → muy similares
- cercano a **0** → poco relacionados
- cercano a **-1** → opuestos (poco común en este tipo de embeddings promedio)


In [ ]:
sim_matrix = cosine_similarity(doc_vectors)

sim_df = pd.DataFrame(sim_matrix, index=df["presidente"], columns=df["presidente"])
sim_df.iloc[:5, :5]


In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(sim_df, cmap="viridis")
plt.title("Heatmap de similaridad entre planes de gobierno")
plt.show()


In [ ]:
# Pares de candidatos más similares
pares = []

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        pares.append({
            "candidato_1": df.loc[i, "presidente"],
            "candidato_2": df.loc[j, "presidente"],
            "similaridad": sim_matrix[i, j]
        })

pares_similares = pd.DataFrame(pares).sort_values("similaridad", ascending=False)
pares_similares.head(10)


# 14. PCA de documentos

Visualizamos ahora los **planes completos** proyectados a 2 dimensiones.


In [ ]:
pca_docs = PCA(n_components=2, random_state=42)
coords_pca = pca_docs.fit_transform(doc_vectors)

plt.figure(figsize=(12, 8))
sns.scatterplot(x=coords_pca[:, 0], y=coords_pca[:, 1])

for i, name in enumerate(df["presidente"]):
    plt.text(coords_pca[i, 0], coords_pca[i, 1], name.split()[0], fontsize=9)

plt.title("Mapa PCA de candidatos")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


# 15. t-SNE de documentos

## ¿Qué es t-SNE?

**t-SNE (t-Distributed Stochastic Neighbor Embedding)** es una técnica no lineal de reducción de dimensionalidad.

A diferencia de PCA, t-SNE intenta preservar mejor la **estructura local**, por eso suele separar mejor clusters en visualización.

### Importante

t-SNE es excelente para **exploración visual**, pero no se debe interpretar como una prueba definitiva de separación ideológica.


In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=5,
    random_state=42,
    init="pca",
    learning_rate="auto"
)

coords_tsne = tsne.fit_transform(doc_vectors)

plt.figure(figsize=(12, 8))
sns.scatterplot(x=coords_tsne[:, 0], y=coords_tsne[:, 1])

for i, name in enumerate(df["presidente"]):
    plt.text(coords_tsne[i, 0], coords_tsne[i, 1], name.split()[0], fontsize=9)

plt.title("Mapa semántico t-SNE de planes de gobierno")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.show()


# 16. Clustering con K-Means

## Definición

**K-Means** es un algoritmo de agrupamiento no supervisado que busca dividir los datos en \(K\) grupos minimizando la distancia entre los puntos y el centroide del cluster.

### Objetivo

\[
\sum ||x_i - \mu_k||^2
\]

donde:

- \(x_i\): observación
- \(\mu_k\): centroide del cluster

En nuestro caso, cada observación es un **plan de gobierno representado como vector**.


In [ ]:
# Método del codo (opcional)
inertias = []

for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(doc_vectors)
    inertias.append({"k": k, "inertia": km.inertia_})

inertias_df = pd.DataFrame(inertias)
inertias_df


In [ ]:
plt.figure(figsize=(7, 4))
sns.lineplot(data=inertias_df, x="k", y="inertia", marker="o")
plt.title("Método del codo para K-Means")
plt.xlabel("Número de clusters")
plt.ylabel("Inercia")
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(doc_vectors)

df[["presidente", "cluster"]].head()


In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=coords_tsne[:, 0],
    y=coords_tsne[:, 1],
    hue=df["cluster"],
    palette="tab10",
    s=80
)

for i, name in enumerate(df["presidente"]):
    plt.text(coords_tsne[i, 0], coords_tsne[i, 1], name.split()[0], fontsize=9)

plt.title("Clusters de candidatos sobre mapa t-SNE")
plt.xlabel("Dimensión 1")
plt.ylabel("Dimensión 2")
plt.show()


# 17. Topic Modeling con LDA

## ¿Qué es Topic Modeling?

Topic Modeling es un conjunto de técnicas para descubrir **temas latentes** en una colección de documentos.

## ¿Qué es LDA?

**Latent Dirichlet Allocation (LDA)** es un modelo probabilístico que asume que:

- cada documento es una mezcla de temas
- cada tema es una distribución de palabras

Esto permite encontrar temas como:

- economía
- salud
- seguridad
- educación
- infraestructura

sin definirlos manualmente.


In [ ]:
count_vectorizer = CountVectorizer(
    max_features=2000,
    min_df=2
)

X_count = count_vectorizer.fit_transform(df["texto_clean"])

lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

X_topics = lda.fit_transform(X_count)

X_topics.shape


In [ ]:
def mostrar_top_palabras_lda(model, feature_names, n_top_words=10):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_features_idx = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_features_idx]
        topics.append({
            "topic": f"Topic {topic_idx}",
            "top_words": ", ".join(top_words)
        })
    return pd.DataFrame(topics)

feature_names_count = count_vectorizer.get_feature_names_out()
topics_df = mostrar_top_palabras_lda(lda, feature_names_count, n_top_words=12)
topics_df


## Interpretación de tópicos

Los tópicos no vienen con nombre automático; el analista debe interpretarlos a partir de las palabras dominantes.

Por ejemplo, un tópico con palabras como:

- salud
- hospital
- médico
- atención

probablemente se interprete como **tema de salud pública**.


In [ ]:
topic_cols = [f"topic_{i}" for i in range(X_topics.shape[1])]
topics_proportions = pd.DataFrame(X_topics, columns=topic_cols)

df_topics = pd.concat([df[["presidente"]], topics_proportions], axis=1)
df_topics.head()


In [ ]:
df["topic_dominante"] = X_topics.argmax(axis=1)

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="topic_dominante")
plt.title("Distribución de tema dominante por candidato")
plt.xlabel("Topic dominante")
plt.ylabel("Cantidad de candidatos")
plt.show()


# 18. WordCloud por candidato

Podemos generar una WordCloud para un candidato específico.

Esto sirve para análisis exploratorio o presentaciones en clase.


In [ ]:
candidato_idx = 0
candidato_nombre = df.loc[candidato_idx, "presidente"]
texto_candidato = df.loc[candidato_idx, "texto_clean"]

wc_candidato = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    collocations=False
).generate(texto_candidato)

plt.figure(figsize=(14, 7))
plt.imshow(wc_candidato, interpolation="bilinear")
plt.axis("off")
plt.title(f"WordCloud – {candidato_nombre}")
plt.show()


# 19. Comparación conceptual: TF-IDF vs Word2Vec

## TF-IDF

**Ventajas**
- simple
- interpretable
- muy útil para clasificación y ranking
- identifica términos distintivos

**Limitaciones**
- no captura contexto semántico profundo
- genera vectores dispersos
- no entiende que palabras parecidas pueden tener significado cercano

## Word2Vec

**Ventajas**
- captura similitud semántica
- produce vectores densos
- permite analogías y proximidad semántica

**Limitaciones**
- requiere suficiente texto
- un promedio de embeddings puede perder información
- una palabra tiene un solo vector, sin contexto dinámico

## Idea clave

- Usa **TF-IDF** cuando quieres resaltar términos importantes
- Usa **Word2Vec** cuando quieres capturar relaciones semánticas


# 20. Limitaciones del enfoque

Es importante enseñar también las limitaciones del pipeline:

1. La limpieza puede eliminar información útil.
2. No estamos haciendo lematización.
3. Word2Vec promedio es una representación simple de documentos.
4. t-SNE es exploratorio, no concluyente.
5. LDA depende mucho de preprocesamiento y número de tópicos.
6. La interpretación política requiere criterio humano; el modelo no reemplaza análisis sustantivo.


# 21. Conclusiones finales

En esta masterclass construimos un pipeline completo de NLP aplicado a datos reales en español.

## Lo que aprendimos

- cómo limpiar texto político
- cómo explorar frecuencia léxica
- cómo representar documentos con TF-IDF
- cómo entrenar embeddings con Word2Vec
- cómo comparar candidatos por similaridad semántica
- cómo visualizar documentos con PCA y t-SNE
- cómo descubrir temas latentes con LDA
- cómo agrupar candidatos con K-Means

## Mensaje final

El verdadero valor del NLP no está solo en ejecutar modelos, sino en **transformar lenguaje humano en estructuras analizables**, conservando al mismo tiempo una interpretación crítica del contexto.


# 22. Ejercicios propuestos para el alumno

1. Construye una lista propia de stopwords en español político y repite TF-IDF.
2. Cambia `vector_size`, `window` y `min_count` en Word2Vec y compara resultados.
3. Prueba distintos valores de `n_components` en LDA.
4. Compara PCA vs t-SNE en términos de interpretación visual.
5. Repite clustering con 3, 4 y 5 clusters y analiza estabilidad.
6. Genera la WordCloud para distintos candidatos y compara vocabulario.
7. Encuentra los 5 pares de candidatos más similares y explica por qué.
